## 1.1. Heat-Neumann Equation

Solve the 1D heat equation: $u_t = \alpha u_{xx}$ for $x$ in [0, 1], $t$ in [0, 1], with initial condition $u(x, 0) = \cos(\pi x)$, boundary conditions $u_x(0,t) = 0$, $u_x(1,t) = 0$.

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
alpha = 1e-2
domain_x = [0, 1]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    u = model(inputs)
    u_t = tape.gradient(u, t)
    u_x = tape.gradient(u, x)
  u_xx = tape.gradient(u_x, x)
  del tape
  return u_t - alpha * u_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t)
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    u_left = model(inputs_left)
    u_right = model(inputs_right)
  u_x_left = tape.gradient(u_left, x_left)
  u_x_right = tape.gradient(u_right, x_right)
  del tape
  return tf.reduce_mean(
      tf.square(u_x_left)
  ) + tf.reduce_mean(
      tf.square(u_x_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_t[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  u = model(inputs)
  u_ic = tf.cos(np.pi * x)
  return tf.reduce_mean(tf.square(u-u_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    print(f"Epoch {epoch}, Loss: {loss.numpy()}")

Epoch 0, Loss: 1.3552775382995605
Epoch 50, Loss: 0.46752089262008667
Epoch 100, Loss: 0.34815314412117004
Epoch 150, Loss: 0.44675126671791077
Epoch 200, Loss: 0.4836103320121765
Epoch 250, Loss: 0.45263534784317017
Epoch 300, Loss: 0.4513709843158722
Epoch 350, Loss: 0.49629098176956177
Epoch 400, Loss: 0.40890011191368103
Epoch 450, Loss: 0.44995811581611633
Epoch 500, Loss: 0.35141417384147644
Epoch 550, Loss: 0.29773539304733276
Epoch 600, Loss: 0.23434026539325714
Epoch 650, Loss: 0.12179990857839584
Epoch 700, Loss: 0.037345655262470245
Epoch 750, Loss: 0.020723100751638412
Epoch 800, Loss: 0.017203878611326218
Epoch 850, Loss: 0.012876664288341999
Epoch 900, Loss: 0.010251166298985481
Epoch 950, Loss: 0.008726110681891441


In [4]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  u_plot = model(inputs_plot).numpy()
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': u_plot.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='u')
fig = px.line(df_melted, x='x', y='u', animation_frame='Time', color='Solution Type',
              title='1D Heat Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='u(x, t)'
)
fig.show()